In [1]:
import torch
import numpy as np

from neural_lam import config
from neural_lam.netCDF_output import initialize_output, edit_netCDF4_attributes, get_netCDF4_timestamp
from neural_lam.netCDF_dataset import NetCDFDataset

In [2]:
def load_pt_batch(pt_data_path, time_idx, var_idx, ensemble_size, size=(400, 550), n_vars=2):
    target = destandardize(
        torch.load(f'{pt_data_path}/example_target_{time_idx}.pt').numpy().reshape(size[0], size[1], n_vars))
    ensemble_mean = destandardize(
        torch.load(f'{pt_data_path}/example_ens_mean_{time_idx}.pt').numpy().reshape(size[0], size[1], n_vars))
    ensemble_std = destandardize(
        torch.load(f'{pt_data_path}/example_ens_std_{time_idx}.pt').numpy().reshape(size[0], size[1], n_vars), std_dataset=True)
    ensemble_members = []
    for ensemble_index in range(ensemble_size):
        ensemble_member = destandardize(
            torch.load(f'{pt_data_path}/example_ens_members_{time_idx}_{ensemble_index}.pt').numpy().reshape(size[0], size[1], n_vars))
        ensemble_members.append(ensemble_member)

    return target, ensemble_mean, ensemble_std, ensemble_members

In [3]:
def destandardize(
    sample,
    pr_stats_path='/mimer/NOBACKUP/groups/mlhighres/projects/detex/HCLIM_EC-Earth3-Veg/standardized_new/mean_std_remapped.pr_EUR-11_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951-2014_mm_day_noleap.npy', 
    tas_stats_path='/mimer/NOBACKUP/groups/mlhighres/projects/detex/HCLIM_EC-Earth3-Veg/standardized_new/mean_std_remapped.tas_EUR-11_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951-2014_noleap.npy',
    std_dataset=False
):
    pr_mean, pr_std = np.load(pr_stats_path)
    tas_mean, tas_std = np.load(tas_stats_path)

    if std_dataset:
        return sample * np.array([pr_std, tas_std])
    else:
        return sample * np.array([pr_std, tas_std]) + np.array([pr_mean, tas_mean])

In [4]:
def write_to_netCDF4(data, netCDF4_dataset, variable_standard_name, var_index, time_idx, input_timestamps):
    for var_name in netCDF4_dataset.variables:
        var = netCDF4_dataset.variables[var_name]
        if var.name == variable_standard_name:
            var[time_idx, :, :] = data[:, :, var_index]
        if var_name == 'time':
            var[time_idx] = get_netCDF4_timestamp(
                input_timestamps, netCDF4_dataset, time_idx)
    netCDF4_dataset.sync()

In [6]:
config_loader = config.Config.from_file("/mimer/NOBACKUP/groups/mlhighres/users/mikhaili/neural-lam/neural_lam/clim_config_inference.yaml")
pt_data_path = '/mimer/NOBACKUP/groups/mlhighres/users/mikhaili/neural-lam/output/val_r1_4_members/pt'
output_path = '/mimer/NOBACKUP/groups/mlhighres/users/mikhaili/neural-lam/output/val_r1_4_members'
model_name = 'SI'
start_date = config_loader.dataset.validation_start_date
end_date = config_loader.dataset.validation_end_date
ensemble_size = 4
batch_size = 1
num_workers = 16
var_index = 1

#variable_name = 'pr'
#variable_standard_name = 'pr'
#variable_long_name = 'pr'
#variable_units = 'kg m-2'

variable_name = 'tas'
variable_standard_name = 'tas'
variable_long_name = 'tas'
variable_units = 'K'

In [7]:
inference_dataloader = torch.utils.data.DataLoader(
    NetCDFDataset(
        start_date=start_date,
        end_date=end_date,
        input_path=config_loader.dataset.input_path,
        input_files=config_loader.dataset.input_files,
        ground_truth_path=config_loader.dataset.ground_truth_path,
        ground_truth_files=config_loader.dataset.ground_truth_files,
        ground_truth_stats_path=config_loader.dataset.ground_truth_stats_path,
        levels=config_loader.dataset.levels,
        is_inference_dataset=False,
        normalize_ground_truth=config_loader.dataset.normalize_ground_truth,
        subset_ds=False,
    ),
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
)

input_files: ['standardized.clt_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.hus_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.pr_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_mm_day_noleap.nc', 'standardized.psl_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.tas_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.ta_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.ua_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.va_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc']


In [8]:
input_datasets = inference_dataloader.dataset.input_datasets
ground_truth_datasets = inference_dataloader.dataset.ground_truth_datasets
input_timestamps = input_datasets[0]['time']

In [9]:
netCDF4_dataset_ensemble_mean = initialize_output(
    inference_dataloader, output_path,
    start_date, end_date, f"{variable_name}_ensemble_mean_{model_name}", var_index
)

edit_netCDF4_attributes(
    netCDF4_dataset_ensemble_mean,
    variable_name,
    variable_standard_name,
    variable_long_name,
    variable_units)

In [10]:
netCDF4_dataset_ensemble_std = initialize_output(
    inference_dataloader, output_path,
    start_date, end_date, f"{variable_name}_ensemble_std_{model_name}", var_index
)

edit_netCDF4_attributes(
    netCDF4_dataset_ensemble_std,
    variable_name,
    variable_standard_name,
    variable_long_name,
    variable_units)

In [11]:
netCDF4_dataset_target = initialize_output(
    inference_dataloader, output_path,
    start_date, end_date, f"{variable_name}_target_{model_name}", var_index
)

edit_netCDF4_attributes(
    netCDF4_dataset_target,
    variable_name,
    variable_standard_name,
    variable_long_name,
    variable_units)

In [12]:
netCDF4_dataset_ensemble_members = []
for ensemble_index in range(ensemble_size):
    netCDF4_dataset_ensemble_member = initialize_output(
        inference_dataloader, output_path,
        start_date, end_date, f"{variable_name}_ensemble_member_{ensemble_index}_{model_name}", var_index
    )
    edit_netCDF4_attributes(
        netCDF4_dataset_ensemble_member,
        variable_name,
        variable_standard_name,
        variable_long_name,
        variable_units)

    netCDF4_dataset_ensemble_members.append(netCDF4_dataset_ensemble_member)

In [13]:
for time_idx in range(len(input_timestamps)):
    target, ensemble_mean, ensemble_std, ensemble_members = load_pt_batch(
        pt_data_path, time_idx, var_index, ensemble_size
    )
    write_to_netCDF4(ensemble_mean, netCDF4_dataset_ensemble_mean, variable_standard_name, var_index, time_idx, input_timestamps)
    write_to_netCDF4(ensemble_std, netCDF4_dataset_ensemble_std, variable_standard_name, var_index, time_idx, input_timestamps)
    write_to_netCDF4(target, netCDF4_dataset_target, variable_standard_name, var_index, time_idx, input_timestamps)
    for ensemble_member_data, netCDF4_dataset_ensemble_member in zip(ensemble_members, netCDF4_dataset_ensemble_members):
        write_to_netCDF4(ensemble_member_data, netCDF4_dataset_ensemble_member, variable_standard_name, var_index, time_idx, input_timestamps)

netCDF4_dataset_ensemble_mean.close()
netCDF4_dataset_ensemble_std.close()
netCDF4_dataset_target.close()
for netCDF4_dataset_ensemble_member in netCDF4_dataset_ensemble_members:
    netCDF4_dataset_ensemble_member.close()